In [ ]:
# Importing libraries
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
# Loading dataset
df = pd.read_csv("HousePricePrediction.csv")
print(df.shape)
df.head()

In [ ]:
pd.set_option('display.max_column', None)
pd.set_option('display.max_rows', None)
# Checking for data types
print(df.dtypes)

In [ ]:
# Converting 'MSZoning', 'LotConfig', 'BldgType' and 'Exterior1st' columns into category datatype
df[['MSZoning', 'LotConfig', 'BldgType', 'OverallCond', 'Exterior1st']] = df[[
    'MSZoning', 'LotConfig', 'BldgType','OverallCond','Exterior1st']].astype('category')
print(df.dtypes)

In [ ]:
# Checking for missing values
df.isna().sum()

In [ ]:
# Filling missing values
df['MSZoning'] = df['MSZoning'].fillna(df['MSZoning'].mode()[0])
df['Exterior1st'] = df['Exterior1st'].fillna(df['Exterior1st'].mode()[0])
df['BsmtFinSF2'] = df['BsmtFinSF2'].fillna(df['BsmtFinSF2'].mean())
df['TotalBsmtSF'] = df['TotalBsmtSF'].fillna(df['TotalBsmtSF'].median())

print("No more missing values")

In [ ]:
# getting dummies of categorical columns 
df = pd.get_dummies(df, drop_first=True)

df.head()

In [ ]:
# Dropping irrelevant columns
df = df.drop(columns= ['Id', 'LotArea', 'BsmtFinSF2'])

In [ ]:
# Checking distribution of target variable
sns.histplot(df['SalePrice'], bins=30)

In [ ]:
# Looking for outliers in target variable
sns.boxplot(y=df['SalePrice'])

In [ ]:
# Handling outliers
df['clipped_sales'] = df['SalePrice'].clip(upper=300000.0)
df[['SalePrice', 'clipped_sales']].describe()

In [ ]:
# Visualizing after using clipping method
sns.histplot(df['clipped_sales'], bins=30)

In [ ]:
# Comparing before and after-effects of clipping method
sns.boxplot(data=df[['clipped_sales', 'SalePrice']])

In [ ]:
# Using winsorization method to further normalize data
from scipy.stats.mstats import winsorize
df["SalePrice_winsor"] = winsorize(df["clipped_sales"], limits=[0, 0.4])

In [ ]:
# Looking for a better normalization technique
df[["clipped_sales", "SalePrice_winsor"]].describe()

In [ ]:
# Comparing both methods
sns.boxplot(data=df[['clipped_sales', 'SalePrice_winsor']])

In [ ]:
# Removing un-normalized and unnecessary target columns
df = df.drop(columns=['SalePrice', 'SalePrice_winsor'])

In [ ]:
# Splitting dataset into test and train dataframes
train_df = df[:1460]
test_df = df[1460:]

In [ ]:
# Splitting Train_df into test, train subsets for better model evaluation
from sklearn.model_selection import train_test_split
X = train_df.drop('clipped_sales', axis=1)
y = train_df['clipped_sales']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Implying Linear Regression model
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
# Comparing actual and predicted values
y_pred = model.predict(X_test)
comparison_df = pd.DataFrame({
    'Original_Value' : y_test.values,
    'Predicted_Value' : y_pred
}, index=X_test.index)

comparison_df.head()

In [ ]:
# Checking model accuracy
model.score(X_test, y_test)